In [48]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from time import sleep
from collections import deque
from itertools import count
from typing import Any, Dict, List, Optional, Tuple, Set

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapperLru import EnvWrapper

from RL.Adapters import FeatureAdapter, NetworkAdapter

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [ ]:
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey  # (vid, layer, tile, gop)

cfg = config.Config()

cfg.filename = f"lfu.csv"
cfg.n_episodes = 100

debugger = debugger.debug

In [ ]:
class LfuPolicy(CachePolicy):

    def __init__(self, max_videos: int = 50, cfg: Any = None) -> None:
        self.cfg = cfg
        self.cur_size = 0
        self.video_capacity = max_videos

        # Video LFU counters.
        self.video_frequency: Dict[int, int] = {}

        # Tile LFU counters per (video, tile).
        self.tile_frequency: Dict[Tuple[int, int], int] = {}

        # Recency list used to break LFU ties: oldest first.
        self.video_access_order: List[int] = []

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def put(
        self,
        key: int,
        value: Any,
        size: int
    ):
        new_video, _ = value

        if new_video in self.video_idx:
            self.video_frequency[new_video] = self.video_frequency.get(new_video, 0) + 1
            self.video_access_order.remove(new_video)
            self.video_access_order.append(new_video)
            return []

        slot = self.video_idx.index(-1) if -1 in self.video_idx else None
        if slot == None:
            min_freq = min(self.video_frequency[v] for v in self.video_idx if v != -1)
            lfu_candidates = {v for v in self.video_idx if v != -1 and self.video_frequency[v] == min_freq}

            lfu_video = next(v for v in self.video_access_order if v in lfu_candidates)
            self.video_access_order.remove(lfu_video)
            self.video_frequency.pop(lfu_video, None)
            slot = self.video_idx.index(lfu_video)

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = [-1] * self.cfg.viewport

        self.video_frequency[new_video] = 1
        self.video_access_order.append(new_video)

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return []

    def put_enhancement_tile(self, key: int, value: Any, size: int):
        new_video, tile = value
                
        self.tile_frequency[(new_video, tile)] = self.tile_frequency.get((new_video, tile), 0) + 1
        
        if new_video in self.video_idx:
            slot = self.video_idx.index(new_video)
            
            historical_tiles = [
                t for (vid, t) in self.tile_frequency.keys() if vid == new_video
            ]
            candidates = set(historical_tiles + [tile])
            
            # Rank by frequency (descending), break ties by tile ID
            ranked_tiles = sorted(
                candidates,
                key=lambda tile: (
                    -self.tile_frequency.get((new_video, tile), 0), tile
                )
            )
            
            # Select top viewport tiles and pad with -1
            selected_tiles = ranked_tiles[:self.cfg.viewport]
            self.tile_idx[slot] = selected_tiles + ([-1] * (self.cfg.viewport - len(selected_tiles)))

    def get(self, key: CacheKey) -> Optional[Any]:
        raise NotImplementedError("LFU policy does not support manual retrieval of individual keys.")

    def contains(self, key: CacheKey) -> bool:
        raise NotImplementedError("LFU policy does not support manual checking of individual keys.")

    def keys(self):
        raise NotImplementedError("LFU policy does not support manual retrieval of keys.")

    def remove(self, key: CacheKey) -> bool:
        raise NotImplementedError("LFU policy does not support manual removal of individual keys.")

    def get_capacity(self) -> int:
        return self.cur_size

    def clear(self) -> None:
        self.video_frequency.clear()
        self.video_access_order.clear()
        self.cur_size = 0
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [ ]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon',
            'lr'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'lr': f"{agent.scheduler.get_last_lr()[0]:.10f}" if agent else None,
            'epsilon': round(float(agent.epsilon), 4) if agent else None
        })

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, (enh_hits + base_hits), (enh_misses + base_misses), base_hits, base_misses

def build_latency_model(cfg):
    """Build and return the MultiDULatencyModel."""
    P = cfg.n_nodes
    max_U = cfg.n_users

    return MultiDULatencyModel(
        P=P,
        max_U=max_U,
        R_M_D=80e6,
        R_C_M=125e6,
        mu=2e7,
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),
        rhoT_p=[0.2],
        lambda_p=[0.05],
        du_fixed_delay=0.001,
        mec_fixed_delay=0.005,
        cloud_fixed_delay=0.1
    )

def build_environment(cfg):
    """Construct the full multi-component environment wrapper."""
    du_caches = []

    policy=LfuPolicy(
        cfg=cfg,
        max_videos=cfg.cache_size,
    )

    # MEC Cache Engine
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity,
        policy=policy
    )

    # User request generator
    users_env = UserRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        arrival_rate=cfg.arrival_rate,
        zipf_alpha=cfg.zipf_alpha
    )

    # Latency Model
    latency_model = build_latency_model(cfg)
    
    # Wrapping all into the main training environment
    return EnvWrapper(
        cfg=cfg,
        n=cfg.n,
        m=cfg.m,
        n_layers=cfg.n_layers,
        users_env=users_env,
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=latency_model,
        theta=cfg.theta,
        lam=cfg.lam,
        max_steps=cfg.max_steps,
        prefetch_fn=lambda cache, action: cache.lru_live_prefetching(action),
        reward_fn=lambda env, reqs: env.compute_reward(reqs),
        debugger=debugger
    )

In [ ]:
def run_episode(episode, env, net_adapter, cfg):
    """Run one full training episode."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0

    env.warmup_phase(net_adapter)

    for step in range(cfg.max_steps):

        req_state = info.get("user_request", None)
        
        _, reward, _, info = env.step(req_state, net_adapter)

        delta_r, hits, misses, bs_hits, bs_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += hits
        cache_misses += misses
        base_hits += bs_hits
        base_misses += bs_miss

        if net_adapter.env_is_done():
            break

        debugger.log('cache_hits', hits)
        debugger.log('cache_misses', misses)

        # print(f"Episode {episode} | Step {step} | Reward: {reward:.2f} | Total Reward: {total_reward:.2f} | Hits: {cache_hits} | Misses: {cache_misses}")

    return total_reward, cache_hits, cache_misses, base_hits, base_misses

def train(cfg):
    print("\n--- Starting DRL Caching System ---")

    env = build_environment(cfg)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    os.makedirs(debug_path, exist_ok=True)

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss = run_episode(
            episode, env, net_adapter, cfg
        )

        save_training_results(
            path_=cfg.path_results + "/" + date_dir,
            filename=cfg.filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=None
        )

        debug_path = os.path.join(cfg.path_results, date_dir)
        os.makedirs(debug_path, exist_ok=True)

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print(
            f"--- Episode {episode} | "
            f"R: {total_reward:.2f} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} ---"
        )
        print("-" * 50)

if __name__ == "__main__":
    train(cfg)


--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 125 videos, 4 tiles per video
--- Episode 0 | R: 0.00 | HR: 0.69 | BHR: 0.88 ---
--------------------------------------------------
--- Episode 1 | R: 0.00 | HR: 0.57 | BHR: 0.73 ---
--------------------------------------------------
--- Episode 2 | R: 0.00 | HR: 0.73 | BHR: 0.93 ---
--------------------------------------------------
--- Episode 3 | R: 0.00 | HR: 0.71 | BHR: 0.90 ---
--------------------------------------------------
--- Episode 4 | R: 0.00 | HR: 0.52 | BHR: 0.66 ---
--------------------------------------------------
--- Episode 5 | R: 0.00 | HR: 0.72 | BHR: 0.91 ---
--------------------------------------------------
--- Episode 6 | R: 0.00 | HR: 0.64 | BHR: 0.81 ---
--------------------------------------------------
--- Episode 7 | R: 0.00 | HR: 0.65 | BHR: 0.82 ---
--------------------------------------------------
--- Episode 8 | R: 0.00 | HR: 0.70 | BHR: 0.89 ---
------------------------

In [54]:
# ----------------------------------------------------------------      
# 1. Publication Style Configuration
# ----------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True
})

# Professional color for line plots
PRIMARY_COLOR = "#2b7bba"

# ----------------------------------------------------------------
# 2. Data Loading & Smoothing
# ----------------------------------------------------------------
path = cfg.path_results + "/" + cfg.filename

print(f"Loading data from: {path}")

df = pd.read_csv(path)

total = (df["cache_hits"] + df["cache_misses"]).replace(0, np.nan)
df["hit_rate"] = (df["cache_hits"] / total) * 100
df["miss_rate"] = (df["cache_misses"] / total) * 100

# Metrics to plot
metrics = ["total_reward", "hit_rate", "miss_rate", "epsilon", "lr"]
window_size = 10  # Adjust smoothing window as needed

# ----------------------------------------------------------------
# 3. Plotting logic
# ----------------------------------------------------------------
# Adjusted figsize for a 4-column row (standard for full-width paper figures)
fig, axes = plt.subplots(1, len(metrics), figsize=(12, 3), sharex=True)

for ax, col in zip(axes, metrics):
    # Plot raw data with transparency (alpha)
    ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, alpha=0.3, linewidth=0.8, label='Raw')
    
    # Plot moving average for clearer trend (except for epsilon which is usually linear)
    if col != "epsilon":
        smoothed = df[col].rolling(window=window_size).mean()
        ax.plot(df["episode"], smoothed, color=PRIMARY_COLOR, linewidth=1.5, label='Trend')
    else:
        # Just a solid line for Epsilon
        ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, linewidth=1.5)

    # Stylistic cleanup
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("Episode")
    
    # Remove redundant Y-labels to save space, or keep for clarity
    ax.set_ylabel("Value") 
    
    series = df[col].dropna()
    if not series.empty:
        ymin = series.min()
        ymax = series.max()
        pad = (ymax - ymin) * 0.05 if ymax != ymin else 1.0
        ax.set_ylim(ymin - pad, ymax + pad)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Tufte-style: remove top/right spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

# Optional: Add a single legend to the first plot if needed
# axes[0].legend(frameon=False)

plt.show()
fig.savefig("drl_caching_metrics.png", dpi=300)

Loading data from: /home/eduardo/Workspace/CacheVideoPredict360/Results/lru.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/eduardo/Workspace/CacheVideoPredict360/Results/lru.csv'